# Cadre de Conception et Justification des Abstractions de Frameworks

## 1. Introduction
La transition d'un prototype d'agent ad-hoc (un script Python brut interpellant l'API d'un fournisseur LLM) vers une application agentique industrielle et testable requiert un changement profond de paradigme. 

En phase d'exploration, assembler des chaînes de caractères et les envoyer en boucle à un modèle de langage (LLM) procure un résultat rapide. Cependant, dès que l'agent doit exécuter des tâches métier critiques, interagir de façon déterministe avec des API tierces, mémoriser des contextes d'utilisateurs distincts sur de longues durées et survivre aux pannes d'exécution, la méthode du script monolithique s'effondre sous le poids de sa propre complexité.

L'**ingénierie des agents** ne consiste pas à ajouter des couches de magie noire ou de dépendances logicielles lourdes. 

Elle repose sur la compréhensionapprofondie des briques élémentaires de l'orchestration logicielle. Ce cours jette les bases architecturales de notre micro-framework propriétaire mini_framework/. Nous allons analyser précisément pourquoi chaque composant existe, comment ils interagissent, et comment se positionner par rapport aux solutions du marché en 2026.

## 2. Où en sommes-nous dans le cursus?

 Le cursus de formation à l'ingénierie IA est structuré selon une progression rigoureuse  :
 - Semaine 1 (Bases et Consolidation) : Maîtrise des architectures de modèles, des patterns de prompts et du raisonnement séquentiel (Few-Shot, Chain-of-Thought).
- Semaine 2 (Agents IA : Fondations & API) : Utilisation pratique du Function Calling via les API d'OpenAI, structuration stricte des sorties JSON, et manipulation d'états de conversation élémentaires.
- Semaine 3 (Multi-agent & protocole MCP) : Spécialisation des rôles d'agents (Planner, Executor, Analyst), coordination multi-agents et introduction au protocole standard d'Anthropic, le Model Context Protocol (MCP).
- Semaine 4 (Frameworks d'Agents - Jour 1 - Aujourd'hui) : Nous prenons de la hauteur. Nous identifions la structure commune à tous les frameworks d'orchestration. Avant de coder notre propre micro-framework demain (Jour 2) et de manipuler les solutions tierces (Jours 3 à 6), nous modélisons les briques clés indispensables d'un système robuste.

## 3. Objectifs pédagogiques

À l'issue de cette journée d'étude et de manipulation de ce support, vous serez capable de : 
- Expliquer pourquoi un framework d'orchestration est nécessaire en production à la place d'une simple boucle de scripts ad-hoc.
- Appliquer le principe de responsabilité unique (SRP) aux composants d'un système logiciel IA.Justifier la séparation stricte entre un agent décideur (cerveau) et les composants de registre d'outils (mains).
- Faire la différence sur le plan de la structure de données entre une conversation à court terme et un système de mémoire persistant à long terme.
- Développer un parseur d'introspection dynamique en Python utilisant le module inspect pour traduire automatiquement du code Python documenté en schémas JSON Schema compatibles avec OpenAI.
- Évaluer les forces et faiblesses architecturales des solutions dominantes du marché en 2026 (LangGraph, CrewAI, OpenAI Agents SDK, Google ADK).

## 4. Pourquoi un framework?

Construire un système agentique performant ne se limite pas à invoquer un modèle de langage avec les bons arguments. En production, un agent doit faire face à des réalités opérationnelles brutales  :
- Gestion des coûts et des limites d'utilisation (Rate Limits) : Suivi précis de la consommation des jetons en entrée et sortie.
- Gestion du temps et de la latence : Parallélisation des appels d'outils, gestion des timeouts et asynchronisme.
- Tolérance aux pannes : Stratégies de ré-exécution (retries) intelligentes si un service externe ou le LLM tombe en panne temporairement.
- Persistance des états : Capacité de mettre en pause l'exécution, de sauvegarder l'état et de le restaurer sans perte d'information.

Un framework d'agents n'est rien d'autre qu'une infrastructure logicielle encapsulant ces défis techniques récurrents. En structurant l'application à l'aide d'abstractions réutilisables, le framework évite de "réinventer la roue" à chaque projet et garantit un code standardisé, évolutif et auditable.

## 5. Histoire des frameworks IA

L'évolution de la stack technologique des agents s'est structurée en plusieurs vagues technologiques rapides  :

- La Vague des "Chaines" (Début 2023) : Popularisée par LangChain, cette approche consistait à enchaîner de manière rigide et séquentielle des prompts prédéfinis ($Prompt_1 \rightarrow LLM_1 \rightarrow Prompt_2 \rightarrow LLM_2$). Ces pipelines déterministes manquaient cruellement d'adaptabilité face aux retours d'observations complexes du monde réel.
- La Vague des "Boucles Autonomes" (Fin 2023) : Projets expérimentaux de type AutoGPT et BabyAGI. Ils ont introduit la boucle d'exécution ReAct (Thought $\rightarrow$ Action $\rightarrow$ Observation) de manière totalement libre. Ces systèmes présentaient de sévères limites d'emballement infini, de surconsommation de jetons et d'imprévisibilité en production.
- La Vague de l'Orchestration d'État (2024-2025) : Émergence de frameworks de graphes comme LangGraph, ou d'équipes structurées comme CrewAI. On ne laisse plus l'agent libre d'aller n'importe où : on borne ses trajectoires au sein d'une machine à états ou d'un graphe orienté cyclique déterministe.
- La Vague de l'Interopérabilité et des Micro-Frameworks (2025-2026) : Standardisation par le biais de protocoles ouverts tels que le Model Context Protocol (MCP) d'Anthropic et le protocole Agent-to-Agent (A2A). On assiste à un retour massif vers des frameworks plus légers, pragmatiques et transparents (comme l'OpenAI Agents SDK) où la couche d'abstraction est réduite à son strict minimum afin de privilégier le contrôle et la traçabilité.

## 6. Quelles responsabilités?
Afin de concevoir une architecture logicielle propre, nous devons appliquer le principe de responsabilité unique (SRP) de Robert C. Martin. Chaque classe ou concept de notre framework doit se concentrer sur une unique source de vérité.
- L'orchestration des flux : Connaître l'ordre des étapes, gérer les transitions conditionnelles et déclencher les exécutions.
- La modélisation de l'état : Assurer la structure et l'intégrité des messages, des variables de contexte et de l'historique d'exécution.
- L'interopérabilité avec les LLM : Traduire les requêtes internes de notre système dans les spécificités des différentes API de modèles et décoder les réponses.
- L'exécution sécurisée des actions : Permettre à l'agent d'agir sur son environnement de manière isolée et supervisée.

## 7. Architecture générale

La structure cible du dépôt Git ai-agent-lab/ que nous construisons pas à pas tout au long de cette formation est conçue pour être modulaire  :

```
ai-agent-lab/
│
├── notebooks/
│   └── S4_J1_intro_framework.ipynb  <-- Le support d'apprentissage interactif d'aujourd'hui
│
├── mini_framework/                  <-- Notre micro-framework propriétaire
│   ├── __init__.py
│   ├── config.py                    <-- Gestion des variables d'environnement, clés et hyperparamètres 
│   ├── message.py                   <-- Modélisation stricte d'un message unitaire 
│   ├── memory.py                    <-- Moteur de persistance court/long terme 
│   ├── tools.py                     <-- Introspection de code Python et abstractions d'outils 
│   ├── registry.py                  <-- Enregistrement et résolution dynamique d'outils 
│   ├── agent.py                     <-- Boucle ReAct et logique décisionnelle de base 
│   └── exceptions.py                <-- Centralisation des erreurs du framework 
│
├── tests/                           <-- Tests automatisés de non-régression
│   └── test_tools.py
│
├── pyproject.toml                   <-- Dépendances du projet (pytest, openai, mcp) 
└── README.md

```
Dans ce paradigme de développement propre, les notebooks servent exclusivement de laboratoires d'expérimentation et de supports pédagogiques. Dès qu'un concept y est éprouvé, son implémentation robuste est exportée sous forme de modules Python testés dans le répertoire mini_framework/.

## 8. Diagramme UML
Voici la modélisation des classes de notre micro-framework et leurs relations logiques au sein du système d'orchestration :

```
             +-----------------------+
             |      AgentConfig      |
             +-----------------------+
             | - model: str          |
             | - temperature: float  |
             | - max_turns: int      |
             +-----------------------+
                         ^
                         | 1
                         |
+------------------------+-----------+
|                   Agent            |
+------------------------------------+
| - config: AgentConfig              |
| - registry: ToolRegistry           |
| - memory: Memory                   |
+------------------------------------+
| + run(user_prompt: str) -> Message |
+------------------+-----------------+
                   | 1
                   |
                   v 1
+------------------+-----------------+
|               Conversation         |
+------------------------------------+
| - messages: List[Message]          |
+------------------------------------+
| + add_message(message: Message)    |
| + get_api_payload() -> List[dict]  |
+------------------+-----------------+
                   | 1
                   |
                   v 1..*
+------------------+-----------------+
|                 Message            |
+------------------------------------+
| + role: str                        |
| + content: str                     |
| + tool_calls: List[dict]           |
+------------------------------------+
```
En parallèle, le sous-système de gestion d'outils s'articule ainsi :

```
+------------------------------------+
|             ToolRegistry           |
+------------------------------------+
| - tools: Dict           |
+------------------------------------+
| + register(func: Callable) -> Tool |
| + get_schemas() -> List[dict]      |
| + execute(name: str, args: dict)   |
+------------------+-----------------+
                   | 1
                   |
                   v 1..*
+------------------+-----------------+
|                  Tool              |
+------------------------------------+
| - func: Callable                   |
| - name: str                        |
| - schema: dict                     |
+------------------------------------+
| + __call__(*args, **kwargs) -> Any |
+------------------------------------+
```

## 9. Diagramme de séquence

Ce diagramme illustre le flux d'interactions au cours d'un tour de décision (ReAct) impliquant un appel d'outil  :
```
Utilisateur        Agent          Registry           Tool            API LLM
     |               |               |                |                 |
     |-- user_msg -->|               |                |                 |
     |               |--------------------- request (messages) -------->|
     |               |                                                  | (Raisonnement)
     |               |<-------------------- request_tool_call ----------|
     |               |                                                  |
     |               |-- check_tool -->|                |                 |
     |               |-- execute(args) -------------->|                 |
     |               |                                | (Exécution)     |
     |               |<------------------- result ----|                 |
     |               |                                                  |
     |               |--------------------- send_observation ----------->|
     |               |                                                  | (Synthèse)
     |               |<-------------------- response_text --------------|
     |<-- answer ----|                                                  |
```

## 10. Pourquoi un Message?


L'abstraction du *Message* est fondamentale pour éviter le couplage technique avec les formats propriétaires des fournisseurs de modèles (OpenAI, Anthropic, Google).

Un message au sein de notre système doit encapsuler uniformément :
- Un rôle : *system* (instructions de base), *user* (requête de l'utilisateur), *assistant* (réponse ou réflexion du modèle), ou *tool* (résultat de l'exécution d'une fonction).
- Un contenu (content) : La chaîne de caractères textuelle brute.
- Des appels d'outils (tool_calls) : Une structure décrivant l'intention du modèle d'interagir avec une fonction spécifique du système.

Modéliser ce message par une classe Python propre (idéalement immuable ou via une Dataclass) permet d'injecter des validateurs d'intégrité de données (vérifier le format des identifiants d'appels d'outils) et de centraliser les fonctions de conversion vers les spécifications des différentes API tierces (par exemple, transformer notre format interne en format d'entrée OpenAI ou en blocs de contenu Anthropic).

## 11. Pourquoi une Conversation?

Une *Conversation* ne doit pas être confondue avec une simple liste de chaînes de caractères. C'est le gestionnaire officiel du contexte à court terme de l'agent.

Elle assume plusieurs rôles essentiels :
- Garantir l'ordre séquentiel : L'API d'un LLM rejette la requête si la séquence des rôles viole les règles logiques d'appel (par exemple, un message de type *tool* doit obligatoirement suivre immédiatement un appel d'outil initié par l' *assistant*).
- Contrôler la dérive de contexte (Token Management) : Si la conversation s'éternise, le nombre total de jetons de l'historique dépasse la fenêtre maximale tolérée par le modèle. C'est au sein de l'objet *Conversation* que sont implémentées les logiques de tronquage dynamique, de fenêtrage glissant (sliding window) ou de résumé automatique (summarization) de l'historique de discussion.

## 12. Pourquoi un Agent?

L' *Agent* est le moteur de raisonnement et de planification du système. C'est une entité sans état (stateless) par nature. Il reçoit un ensemble de directives de comportement, une configuration de modèle, un catalogue d'actions possibles (outils) et l'état courant de la conversation.

L'Agent n'exécute pas directement les requêtes HTTP, n'interagit pas avec la base de données et ne manipule pas le terminal de l'application. Sa responsabilité unique consiste à exécuter la politique du modèle de langage pour décider de la prochaine étape  :
- Est-ce que le problème est résolu? Si oui, il produit la réponse finale adressée à l'utilisateur.
- Faut-il aller chercher une information complémentaire? Si oui, il formule une demande structurée d'appel d'outil.

En gardant l'Agent isolé de l'infrastructure d'exécution directe, on élimine de nombreuses vulnérabilités de sécurité et on facilite le test automatisé de ses capacités de raisonnement.

## 13. Pourquoi une Configuration?

L'un des principaux anti-patterns rencontrés dans l'écriture de code d'agents est le codage en dur ("hardcoding") des hyperparamètres du modèle de langage (le nom du modèle cible, la température de créativité, le budget maximal de jetons, les limites de temps d'attente d'une requête) directement au sein du code de l'orchestrateur.

L'abstraction *AgentConfig* isole ces variables. Elle offre plusieurs avantages :
- La flexibilité d'infrastructure : Permet de basculer instantanément d'un modèle ultra-performant (ex: GPT-4o) à un modèle local compact (ex: Llama 3) selon les contraintes de coût ou de latence, sans modifier une ligne de code logique de l'agent.
- L'optimisation par environnement : Charger des configurations différentes selon que le code s'exécute en local pour du test unitaire (latence réduite) ou en production distribuée (sécurité maximale).

## 14. Pourquoi un Tool?

Un *Tool* (outil) est le pont qui relie le modèle de langage au monde extérieur réel. Il résout une dualité fondamentale  :
- Pour le LLM : Un outil est une pure abstraction de métadonnées documentées. Le modèle a besoin de connaître son nom, sa description textuelle générale, ses arguments attendus et leurs contraintes de formats (JSON Schema).
- Pour le système informatique : Un outil est une fonction Python réelle, exécutable localement ou via un appel réseau.

L'abstraction de la classe Tool encapsule ces deux dimensions en une seule interface propre. Elle utilise l'introspection de code pour générer le schéma JSON requis par le LLM, tout en conservant une référence directe vers la fonction Python d'origine pour l'exécuter le moment venu.

## 15. Pourquoi un Registry?

Au sein d'un framework d'agents digne de ce nom, un outil ne doit jamais être instancié de manière isolée ou dispersée. Le *ToolRegistry* sert de catalogue unique et sécurisé d'actions autorisées.
Ses responsabilités sont capitales en production :
- La ségrégation d'accès : L'agent ne manipule pas l'espace de noms Python global. Il interroge le registre qui filtre et n'expose au modèle de langage que les schémas d'outils configurés pour sa session de travail, évitant la surcharge cognitive et les fuites de contexte.
- La sécurité d'exécution : Le registre joue le rôle de barrière de contrôle. Si le LLM tente de déclencher une injection de commande ou d'exécuter une fonction inexistante (hallucination de nom de fonction), le registre intercepte l'appel, valide les arguments contre le schéma strict JSON Schema et lève une exception logicielle contrôlée au lieu de compromettre le serveur.

## 16. Pourquoi une Memory?

Dans un système conversationnel classique, l'historique des échanges est entièrement renvoyé à chaque requête. Mais cette approche souffre d'une limite physique et financière évidente : la surcharge de mémoire à long terme.

La *Memory* est un composant tiers, indépendant du flux conversationnel immédiat. Son but est de simuler la mémoire humaine à travers différents niveaux de persistance  :

- Mémoire épisodique (court terme) : Conservation du fil de la conversation courante.
- Mémoire sémantique (long terme) : Extraction analytique d'informations de profil utilisateur, de préférences explicites ou de connaissances spécifiques apprises lors de sessions passées, indexées dans une base de données de vecteurs (RAG).

La mémoire sémantique permet d'interroger la base de données de connaissances et d'injecter sélectivement au LLM uniquement les informations clés nécessaires au traitement du tour courant, préservant ainsi la taille de la fenêtre de contexte et améliorant de plus de 90 % la précision logique du modèle.

## 17. Comparaison avec LangChain

LangChain est le pionnier historique du développement d'applications LLM.
```
+-----------------------------------------------------------------+
|                        LangChain                                |
|  Pros : Vaste catalogue d'intégrations prêtes à l'emploi [21]   |
|  Cons : Abstractions trop imbriquées, difficile à déboguer [6]  |
+-----------------------------------------------------------------+
```
**Forces**
- Un écosystème colossal d'intégrations prêtes à l'emploi (bases de données de vecteurs, connecteurs d'API, chargeurs de documents).
- La possibilité de prototyper un agent basique extrêmement rapidement en assemblant des briques pré-packagées.

**Faiblesses**
- Une épaisseur d'abstractions excessivement lourde (chaînes imbriquées).
- Les développeurs de production se heurtent rapidement à un effet "boîte noire" complexe à tracer et à personnaliser lorsque les comportements de l'agent dévient.

## 18. Comparaison avec LangGraph

LangGraph est l'évolution moderne de l'écosystème de LangChain, axée sur la modélisation sous forme de graphes.
```
+-----------------------------------------------------------------+
|                        LangGraph                                |
|  Pros : Graphes d'état cycliques déterministes                  |
|  Cons : Courbe d'apprentissage élevée, verbeux pour les cas     |
|         simples [26, 19]                                        |
+-----------------------------------------------------------------+
```
**Forces**
- Idéal pour modéliser des comportements hautement cycliques complexes nécessitant des retours en arrière.
- Gestion d'état robuste avec checkpointing natif persistant (permettant le "voyage dans le temps" pour relire ou modifier des étapes de décision passées).
- Parfaitement outillé via l'environnement visuel d'analyse LangSmith/LangGraph Studio.

**Faiblesses**
- Courbe d'apprentissage abrupte : nécessite de repenser la logique applicative sous forme de nœuds, d'arêtes logiques et de schémas d'état.
- Très verbeux à implémenter pour des agents linéaires basiques.

## 19. Comparaison avec OpenAI SDK

L'Agents SDK d'OpenAI (conçu à partir des apprentissages du framework expérimental Swarm) propose une vision minimaliste axée sur le code standard.

```
+-----------------------------------------------------------------+
|                      OpenAI Agents SDK                          |
|  Pros : Minimaliste, sans abstractions magiques, handoffs       |
|  Cons : Lié principalement aux modèles OpenAI [12, 27]          |
+-----------------------------------------------------------------+
```
**Forces**
- Une architecture d'une grande élégance basée sur seulement 4 primitives simples (Agents, Handoffs, Guardrails, Tracing).
- Le concept de "handoff" (transfert de contrôle explicite entre agents pairs) élimine la complexité d'un superviseur centralisé.
- Pas d' abstractions magiques cachées : le développeur garde le contrôle complet du code Python d'exécution.

**Faiblesses**
- Optimisé principalement pour les modèles OpenAI (bien que supportant d'autres modèles via LiteLLM).
- Moins d'outils natifs pour les cycles complexes par rapport à LangGraph.

## 20. Comparaison avec CrewAI

CrewAI propose d'orchestrer des agents selon une métaphore humaine de collaboration d'équipe.

```
+-----------------------------------------------------------------+
|                         CrewAI                                  |
|  Pros : Métaphore d'équipe intuitive, prototypage ultra-rapide  |
|                                                                 |
|  Cons : Limité pour la production complexe, pas de graphes      |
+-----------------------------------------------------------------+
```
**Forces**
- API déclarative de très haut niveau, intuitive à appréhender (on instancie des rôles, des backstories d'agents et des tâches à accomplir).
- Idéal pour le prototypage rapide d'équipes de traitement de contenu, de rédaction ou de recherche d'informations en quelques minutes.

**Faiblesses**
- Trop restrictif pour concevoir des applications complexes nécessitant des logiques d'embranchements cycliques déterministes.
- Gestion de la mémoire et résilience aux pannes réseau en production limitée.

## 21. Comparaison avec ADK

L'Agent Development Kit (ADK) de Google est un outil haut de niveau de la stack GCP.
```
+-----------------------------------------------------------------+
|                       Google ADK                                |
|  Pros : Intégration GCP native (Vertex, Cloud Run)              |
|  Cons : Moins universel, écosystème fermé                       |
+-----------------------------------------------------------------+
```

## Forces
- Parfaitement intégré à l'infrastructure infonuagique de Google Cloud Platform (Vertex AI, Cloud Run, Gemini, Agent Engine).
- Support natif des protocoles inter-agents (A2A) et des flux de données multimodaux bidirectionnels audio et vidéo.
## Faiblesses
- Moins de maturité au sein de la communauté open-source en dehors de l'écosystème GCP.
- Orienté principalement vers les architectures d'entreprise rigides.
## 22. Sous le capot : Introspection Dynamique de Code

Pour comprendre concrètement le fonctionnement de la génération dynamique de schémas JSON Schema par introspection, voici le code de référence utilisant le module natif *inspect* de Python  :

In [ ]:
import inspect
import json
from typing import get_type_hints, Callable, Dict, Any

# Table de correspondance standardisée entre types Python et JSON Schema
MAP_TYPES_JSON = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
    type(None): "null"
}

def generer_schema_json(func: Callable[..., Any]) -> Dict[str, Any]:
    """
    Analyse par introspection la signature et la documentation d'une fonction Python
    pour générer une spécification JSON Schema compatible OpenAI.
    """
    # 1. Analyse de la signature de la fonction
    sig = inspect.signature(func)
    type_hints = get_type_hints(func)
    
    # 2. Lecture et extraction de la documentation (docstring)
    docstring = inspect.getdoc(func) or ""
    
    properties = {}
    required =

    # 3. Traitement des paramètres d'entrée
    for nom_param, param in sig.parameters.items():
        if nom_param in ["self", "cls", "context"]:
            continue  # Ignorer les variables systèmes
            
        # Résolution du type du paramètre
        type_python = type_hints.get(nom_param, param.annotation)
        type_json = MAP_TYPES_JSON.get(type_python, "string")
        
        properties[nom_param] = {
            "type": type_json,
            "description": f"Paramètre d'entrée : {nom_param}"
        }
        
        # S'il n'y a pas de valeur par défaut, le paramètre est requis
        if param.default is inspect.Parameter.empty:
            required.append(nom_param)

    # 4. Construction de l'objet final JSON Schema
    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": docstring.split("\n") if docstring else f"Exécute {func.__name__}",
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required
            }
        }
    }

# Démonstration pratique :
def planifier_mcp_session(serveur_url: str, jetons_max: int = 4096) -> bool:
    """Initialise une session de communication avec un serveur MCP externe."""
    return True

schema = generer_schema_json(planifier_mcp_session)
print(json.dumps(schema, indent=2, ensure_ascii=False))

## 23. Notes Marché

L'évaluation pragmatique des solutions logicielles d'IA en 2026 repose sur des arbitrages financiers et techniques clairs  :

- Le Coût Total de Possession (TCO) : Utiliser des protocoles d'intégration massifs comme le Model Context Protocol (MCP) d'Anthropic simplifie la réutilisabilité des outils à travers l'écosystème. Cependant, le protocole standardisé JSON-RPC de MCP est extrêmement verbeux en métadonnées. Ces schémas volumineux saturent rapidement la fenêtre de contexte du LLM, augmentant les coûts de jetons d'entrée jusqu'à 30 % par rapport à des schémas d'outils optimisés de manière personnalisée en Python pur.
- La Sécurité Métier : Les organisations basculent vers des architectures d'agents hautement supervisées (Human-in-the-loop). Les frameworks qui facilitent l'approbation humaine avant l'exécution d'une action à risque (écriture en base de données, envoi de transactions financières) s'imposent par rapport aux agents d'exécution totalement libres.

## 24. Notes d'Architecte

**La Réalité des Abstractions Applicatives**

La majorité des équipes techniques débutent le prototypage d'un agent en utilisant les frameworks les plus populaires (LangChain, CrewAI) en raison de la simplicité des démos en ligne. Mais dès le passage à l'échelle industrielle, les développeurs constatent l'opacité et l'instabilité de ces outils. Face à des pannes survenant en production, remonter la trace de l'erreur au travers de couches d'abstractions magiques et de dizaines de fonctions de rappel (callbacks) s'avère extrêmement coûteux.

La tendance forte en ingénierie logicielle IA consiste à déconstruire ces frameworks pour construire des architectures personnalisées (thin-wrappers) reposant sur du code Python explicite, où la mémoire et l'état de l'agent résident directement dans les bases de données SQL ou Redis plutôt que d'être gérés par un outil tiers opaque.

## 25. Exercices
**Objectif**

Implémenter de manière simple les classes de structure de données de base *Message* et *Conversation* au sein de votre notebook pour comprendre leur rôle d'isolation.

**Consignes**
- Utilisez des Dataclasses Python pour implémenter la classe *Message*.
- Implémentez la classe *Conversation* avec une méthode *ajouter_message*.
- Ajoutez à la classe *Conversation* une fonction *obtenir_payload_openai* qui formate la liste des messages de sorte qu'elle soit directement compréhensible par l'API d'OpenAI.

In [1]:
from dataclasses import dataclass, field
from typing import List, Dict, Any

@dataclass
class Message:
    role: str
    content: str
    tool_calls: List] = field(default_factory=list)

    def to_openai_dict(self) -> Dict[str, Any]:
        """Convertit notre message interne au format d'API OpenAI."""
        payload = {"role": self.role, "content": self.content}
        if self.tool_calls:
            payload["tool_calls"] = self.tool_calls
        return payload

class Conversation:
    def __init__(self):
        self.messages: List[Message] =

    def ajouter_message(self, role: str, content: str, tool_calls: List] = None):
        """Instancie et ajoute un message propre à la conversation."""
        msg = Message(role=role, content=content, tool_calls=tool_calls or)
        self.messages.append(msg)

    def obtenir_payload_openai(self) -> List]:
        """Génère la liste complète des payloads au format OpenAI."""
        return [msg.to_openai_dict() for msg in self.messages]

# Code de Test :
conv = Conversation()
conv.ajouter_message("user", "Quelle est la météo à Paris?")
print(conv.obtenir_payload_openai())

SyntaxError: unmatched ']' (2556961475.py, line 8)

## 26. Défis

**Objectif**

Concevoir un décorateur Python personnalisé nommé *@registre_outil* qui enregistre dynamiquement la fonction décorée dans un registre central tout en lui rattachant automatiquement son schéma JSON généré par introspection.

**Consignes**

Développez la classe *Registry* et le décorateur de fonction pour valider le scénario d'exécution suivant :

In [2]:
class Registry:
    def __init__(self):
        self.catalogue = {}

    def enregistrer(self, func):
        # Utiliser la fonction 'generer_schema_json' de la section 22
        schema = generer_schema_json(func)
        self.catalogue[func.__name__] = {
            "execution": func,
            "schema": schema
        }
        return func

# Instanciation globale
mon_registre = Registry()

# Décorateur
def registre_outil(func):
    return mon_registre.enregistrer(func)

# Test de validation :
@registre_outil
def obtenir_coordonnees(ville: str) -> dict:
    """Récupère les coordonnées géographiques d'une ville ciblée."""
    return {"lat": 48.8566, "lon": 2.3522}

# Validation de l'enregistrement automatique
assert "obtenir_coordonnees" in mon_registre.catalogue
print("Défi validé! Schéma généré dans le catalogue :")
print(json.dumps(mon_registre.catalogue["obtenir_coordonnees"]["schema"], indent=2, ensure_ascii=False))

NameError: name 'generer_schema_json' is not defined

## 27. Synthèse

Le tableau ci-dessous synthétise les apprentissages clés de cette session d'ingénierie applicative :

- Un framework n'est pas magique : C'est une architecture logicielle éprouvée visant à séparer proprement les tâches de décision (Agent), de contexte (Conversation), d'action (Outils) et d'auditabilité (Tracing).
- Le principe SRP s'applique intégralement : L'agent formule une intention logique de calcul, l'environnement applicatif hôte l'exécute de manière sécurisée.
- L'introspection Python libère du temps de développement en maintenant une source unique de vérité logique au sein de votre code Python.

## 28. Bibliographie

Pour aller plus loin et consolider vos compétences d'architecte IA, nous vous recommandons d'étudier les documentations techniques suivantes :

- OpenAI Developer Guides — Function Calling and Structured Outputs :(https://developers.openai.com/api/docs/guides/structured-outputs)
- Anthropic Model Context Protocol (MCP) — Architecture and Transport Protocol Specifications :(https://modelcontextprotocol.io)
- Python Software Foundation — Module inspect: Inspecting Live Objects :(https://docs.python.org/3/library/inspect.html)
- LangGraph Reference Manual — State and Checkpointing Design Patterns :(https://langchain-ai.github.io/langgraph/) 

## 29. Ce qui sera développé demain

Demain (Semaine 4 — Jour 2), nous passerons de la théorie à la pratique dure en production. Nous allons écrire intégralement le code source de notre micro-framework propriétaire *mini_framework/*. 
En moins de 300 lignes de code Python standard robuste et asynchrone, nous allons implémenter  :   

- La boucle d'exécution ReAct complète (agent.py).   

- Le moteur de persistance de session de conversation (memory.py).   

- Le registre d'outils dynamique avec validation d'arguments strictes (registry.py & tools.py).   

À la fin de la journée de demain, vous posséderez votre propre infrastructure de développement opérationnelle et prête à être testée unitairement via pytest.   

## 30. Conclusion
Comprendre les forces de tension architecturale qui régissent la conception d'un framework d'agents est ce qui distingue un développeur de démonstrations (prototypes jetables) d'un véritable ingénieur logiciel IA capable de concevoir des systèmes industriels.   

En refusant d'utiliser aveuglément les bibliothèques lourdes du marché et en concevant vous-même votre propre structure de composants découplés, vous maîtrisez la trajectoire d'exécution, la latence et la sécurité de vos agents. C'est ce socle technique qui fera la valeur de votre profil de compétences.   

